# Day 7: Loss Functions & Optimizers (SGD, Adam, RMSprop)

**Module 2 — Neural Network Basics | 100 Days of Data Science**

## Why This Matters
Day 6 showed the full training loop with plain gradient descent. But in practice, the *choice* of loss function and optimizer has a massive effect on how fast (and whether) a model converges. Today we cover the standard menu of both, and see why Adam became the default choice for most deep learning problems.

## Topics Covered Today
1. Regression losses: MSE vs MAE
2. Classification losses: Binary & Categorical Cross-Entropy
3. Plain SGD — and its weaknesses
4. SGD with Momentum
5. RMSprop
6. Adam
7. Side-by-side optimizer comparison on the same problem
8. Practice exercises

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("NumPy version:", np.__version__)

---
## 1. Regression Losses: MSE vs MAE

**Mean Squared Error (MSE):** $\frac{1}{n}\sum(\hat{y}-y)^2$ — penalizes large errors heavily (squared), smooth gradient, sensitive to outliers.

**Mean Absolute Error (MAE):** $\frac{1}{n}\sum|\hat{y}-y|$ — penalizes all errors linearly, more robust to outliers, but has a non-smooth gradient at 0.

In [ ]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

errors = np.linspace(-5, 5, 200)
mse_curve = errors ** 2
mae_curve = np.abs(errors)

plt.figure(figsize=(6,4))
plt.plot(errors, mse_curve, label='MSE (squared error)')
plt.plot(errors, mae_curve, label='MAE (absolute error)')
plt.xlabel('error (y_pred - y_true)')
plt.ylabel('loss')
plt.title('MSE punishes large errors much more heavily than MAE')
plt.legend()
plt.grid(True)
plt.show()

# Effect of an outlier
y_true = np.array([1, 2, 3, 4])
y_pred_normal = np.array([1.1, 2.1, 2.9, 4.2])
y_pred_outlier = np.array([1.1, 2.1, 2.9, 10.0])  # one bad prediction

print("Normal case  -> MSE:", round(mse(y_true, y_pred_normal), 3), "| MAE:", round(mae(y_true, y_pred_normal), 3))
print("With outlier -> MSE:", round(mse(y_true, y_pred_outlier), 3), "| MAE:", round(mae(y_true, y_pred_outlier), 3))
print("\nMSE explodes with the outlier; MAE grows much more slowly.")

---
## 2. Classification Losses: Binary & Categorical Cross-Entropy

Covered in depth on Day 3. Quick recap + code:

**Binary Cross-Entropy** (2 classes): $-[y\log(\hat{y}) + (1-y)\log(1-\hat{y})]$

**Categorical Cross-Entropy** (multi-class): $-\sum_i y_i \log(\hat{y}_i)$

In [ ]:
def binary_cross_entropy(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-12, 1 - 1e-12)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def categorical_cross_entropy(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-12, 1)
    return -np.sum(y_true * np.log(y_pred))

# Binary example
print("BCE (confident & correct):", binary_cross_entropy(np.array([1]), np.array([0.95])))
print("BCE (confident & wrong):  ", binary_cross_entropy(np.array([1]), np.array([0.05])))

# Multi-class example
true_onehot = np.array([0, 1, 0])
pred_probs = np.array([0.1, 0.8, 0.1])
print("\nCategorical CE:", categorical_cross_entropy(true_onehot, pred_probs))

### Quick Reference

| Task | Loss Function |
|---|---|
| Regression | MSE (default) or MAE (outlier-robust) |
| Binary classification | Binary Cross-Entropy |
| Multi-class classification | Categorical Cross-Entropy |

---
## 3. Plain SGD — and Its Weaknesses

$$w_{new} = w_{old} - \eta \nabla L(w)$$

Simple, but has real problems:
- Same learning rate for every parameter, regardless of how steep or flat that direction is
- Can oscillate badly in narrow "ravines" of the loss surface
- No memory of past gradients — no momentum to push through flat regions or small local bumps

In [ ]:
# A loss surface shaped like a narrow ravine -- classic case where plain SGD struggles
def ravine_loss(w1, w2):
    return 0.1 * w1**2 + 5 * w2**2  # steep in w2 direction, shallow in w1

def ravine_grad(w1, w2):
    return np.array([0.2 * w1, 10 * w2])

def run_sgd(start, lr, steps):
    w = np.array(start, dtype=float)
    path = [w.copy()]
    for _ in range(steps):
        grad = ravine_grad(*w)
        w = w - lr * grad
        path.append(w.copy())
    return np.array(path)

path_sgd = run_sgd(start=[-4, 1], lr=0.15, steps=40)

w1_range = np.linspace(-5, 5, 100)
w2_range = np.linspace(-2, 2, 100)
W1, W2 = np.meshgrid(w1_range, w2_range)
Z = ravine_loss(W1, W2)

plt.figure(figsize=(7,5))
plt.contour(W1, W2, Z, levels=30, cmap='viridis')
plt.plot(path_sgd[:,0], path_sgd[:,1], 'ro-', markersize=3, label='Plain SGD path')
plt.xlabel('w1')
plt.ylabel('w2')
plt.title('Plain SGD oscillates in narrow ravines')
plt.legend()
plt.show()

---
## 4. SGD with Momentum

Adds a "velocity" term that accumulates past gradients, smoothing out oscillations and helping push through flat regions:

$$v_t = \beta v_{t-1} + (1-\beta)\nabla L(w)$$
$$w_{new} = w_{old} - \eta v_t$$

In [ ]:
def run_momentum(start, lr, steps, beta=0.9):
    w = np.array(start, dtype=float)
    v = np.zeros_like(w)
    path = [w.copy()]
    for _ in range(steps):
        grad = ravine_grad(*w)
        v = beta * v + (1 - beta) * grad
        w = w - lr * v
        path.append(w.copy())
    return np.array(path)

path_momentum = run_momentum(start=[-4, 1], lr=0.15, steps=40)

plt.figure(figsize=(7,5))
plt.contour(W1, W2, Z, levels=30, cmap='viridis')
plt.plot(path_sgd[:,0], path_sgd[:,1], 'ro-', markersize=3, label='Plain SGD', alpha=0.6)
plt.plot(path_momentum[:,0], path_momentum[:,1], 'bo-', markersize=3, label='SGD + Momentum')
plt.xlabel('w1')
plt.ylabel('w2')
plt.title('Momentum smooths the path and converges faster')
plt.legend()
plt.show()

---
## 5. RMSprop

Adapts the learning rate **per parameter** by dividing by a running average of recent squared gradients. Directions with large/steep gradients get smaller effective steps; flat directions get relatively bigger steps.

$$s_t = \beta s_{t-1} + (1-\beta) (\nabla L)^2$$
$$w_{new} = w_{old} - \frac{\eta}{\sqrt{s_t} + \epsilon} \nabla L$$

In [ ]:
def run_rmsprop(start, lr, steps, beta=0.9, eps=1e-8):
    w = np.array(start, dtype=float)
    s = np.zeros_like(w)
    path = [w.copy()]
    for _ in range(steps):
        grad = ravine_grad(*w)
        s = beta * s + (1 - beta) * grad**2
        w = w - (lr / (np.sqrt(s) + eps)) * grad
        path.append(w.copy())
    return np.array(path)

path_rmsprop = run_rmsprop(start=[-4, 1], lr=0.3, steps=40)

plt.figure(figsize=(7,5))
plt.contour(W1, W2, Z, levels=30, cmap='viridis')
plt.plot(path_sgd[:,0], path_sgd[:,1], 'ro-', markersize=3, label='Plain SGD', alpha=0.4)
plt.plot(path_momentum[:,0], path_momentum[:,1], 'bo-', markersize=3, label='SGD + Momentum', alpha=0.6)
plt.plot(path_rmsprop[:,0], path_rmsprop[:,1], 'go-', markersize=3, label='RMSprop')
plt.xlabel('w1')
plt.ylabel('w2')
plt.title('RMSprop adapts step size per-parameter')
plt.legend()
plt.show()

---
## 6. Adam (Adaptive Moment Estimation)

Combines **Momentum** (1st moment — mean of gradients) and **RMSprop** (2nd moment — variance of gradients), plus bias correction. This is why Adam is the default optimizer for most deep learning tasks today.

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\nabla L \quad \text{(momentum term)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)(\nabla L)^2 \quad \text{(RMSprop term)}$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t} \quad \text{(bias correction)}$$
$$w_{new} = w_{old} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon}$$

In [ ]:
def run_adam(start, lr, steps, beta1=0.9, beta2=0.999, eps=1e-8):
    w = np.array(start, dtype=float)
    m = np.zeros_like(w)
    v = np.zeros_like(w)
    path = [w.copy()]
    for t in range(1, steps + 1):
        grad = ravine_grad(*w)
        m = beta1 * m + (1 - beta1) * grad
        v = beta2 * v + (1 - beta2) * grad**2
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        w = w - lr * m_hat / (np.sqrt(v_hat) + eps)
        path.append(w.copy())
    return np.array(path)

path_adam = run_adam(start=[-4, 1], lr=0.3, steps=40)

plt.figure(figsize=(7,5))
plt.contour(W1, W2, Z, levels=30, cmap='viridis')
plt.plot(path_sgd[:,0], path_sgd[:,1], 'ro-', markersize=3, label='Plain SGD', alpha=0.35)
plt.plot(path_momentum[:,0], path_momentum[:,1], 'bo-', markersize=3, label='SGD + Momentum', alpha=0.5)
plt.plot(path_rmsprop[:,0], path_rmsprop[:,1], 'go-', markersize=3, label='RMSprop', alpha=0.6)
plt.plot(path_adam[:,0], path_adam[:,1], 'mo-', markersize=3, label='Adam')
plt.xlabel('w1')
plt.ylabel('w2')
plt.title('Adam combines Momentum + RMSprop')
plt.legend()
plt.show()

---
## 7. Side-by-Side: Loss Convergence Comparison

In [ ]:
def loss_along_path(path):
    return [ravine_loss(w1, w2) for w1, w2 in path]

plt.figure(figsize=(7,5))
plt.plot(loss_along_path(path_sgd), label='Plain SGD')
plt.plot(loss_along_path(path_momentum), label='SGD + Momentum')
plt.plot(loss_along_path(path_rmsprop), label='RMSprop')
plt.plot(loss_along_path(path_adam), label='Adam')
plt.yscale('log')
plt.xlabel('Step')
plt.ylabel('Loss (log scale)')
plt.title('Convergence Speed Comparison on the Same Ravine Loss')
plt.legend()
plt.grid(True)
plt.show()

### Using These in PyTorch
In practice you never hand-write these — just pick the optimizer class:
```python
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # most common default
```

---
## 8. Practice Exercises
Try these before Day 8:

1. Change the ravine's steepness (`5 * w2**2` → `20 * w2**2`) and re-run all four optimizers. Which one degrades least?
2. Implement AdaGrad (like RMSprop but accumulates squared gradients WITHOUT decay) and add it to the comparison plot.
3. Try three different learning rates for Adam (`0.01`, `0.3`, `1.0`) — what happens at the extremes?
4. Retrain the Day 6 XOR MLP using momentum instead of plain gradient descent — does it converge faster?
5. In your own words: why does Adam need bias correction ($\hat{m}_t$, $\hat{v}_t$) in the first few steps specifically?

In [ ]:
# Your practice code here


---
## Summary
- **MSE/MAE** for regression; **cross-entropy** for classification
- **Plain SGD** is simple but oscillates and treats all parameters equally
- **Momentum** smooths the path using accumulated past gradients
- **RMSprop** adapts the learning rate per parameter based on recent gradient magnitude
- **Adam** combines both — momentum + adaptive learning rates — and is the most common default optimizer in practice

Next up: **Day 8 — Intro to PyTorch/TensorFlow (building models the standard way)**

---
*Part of the 100 Days of Data Science series | DL-for-Data-Science repo*